# Feature Scaling and Normalization

## 1. Concept Introduction

Feature scaling is a preprocessing step in the machine learning pipeline used to standardize the range of independent variables or features of data. 

- In raw datasets, features often exhibit vastly different units and magnitudes (e.g., age in years vs. income in dollars).
- Many machine learning algorithms, particularly those based on distance measures (like KNN) or gradient-based optimization (like Logistic Regression), are extremely sensitive to these magnitude discrepancies.
- If left unscaled, features with larger numeric ranges disproportionately dominate the model's cost function or distance calculations, effectively drowning out features with smaller scales regardless of their actual predictive importance.

## 2. Intuition Section

Imagine a distance-based algorithm like K-Nearest Neighbors (KNN) trying to classify a customer based on Age (range 18 to 95) and Income (range 20,000 to 500,000).

When the model calculates the distance between two data points using Euclidean distance, the Income variable will dominate the calculation simply because the numbers are much larger. The model will essentially classify based on Income and ignore Age, even if Age is a highly predictive feature. 

Feature scaling levels this playing field. It forces all features into a comparable mathematical space, ensuring the algorithm evaluates the relationships between data points based on their structural proximity rather than the arbitrary magnitude of their units.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries successfully imported and environment initialized.")

## 3. Creating the Synthetic Dataset

To properly demonstrate the impact of feature scaling, we need to create our own synthetic dataset right here in the notebook. We will generate two features:
- Age: Ranges roughly from 18 to 80.
- Income: Ranges roughly from 30,000 to 150,000.

We will also create a target variable (Bought_Product) that relies on BOTH features. This ensures that a good model must pay attention to both Age and Income to make accurate predictions.

In [ ]:
# Generate 1000 synthetic data points
n_samples = 1000

# Feature 1: Age (Normally distributed around 45)
age = np.random.normal(loc=45, scale=12, size=n_samples)
age = np.clip(age, 18, 90) # Keep within realistic bounds

# Feature 2: Income (Normally distributed around 80,000)
income = np.random.normal(loc=80000, scale=25000, size=n_samples)
income = np.clip(income, 20000, 250000)

# Create the target variable: 1 if they bought the product, 0 otherwise.
# We define a hidden relationship where older people and higher earners buy more.
z_age = (age - 45) / 12
z_income = (income - 80000) / 25000
probability = 1 / (1 + np.exp(-(z_age + z_income)))
bought_product = np.random.binomial(n=1, p=probability)

# Assemble into a Pandas DataFrame
df = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Bought_Product': bought_product
})

print("Synthetic dataset generated successfully!")

In [ ]:
# Display data distribution and summary
print("--- Data Head ---")
print(df.head())

print("\n--- Summary Statistics ---")
print(df.describe().round(2))

print("\nNotice the massive difference in magnitude between the mean Age (~45) and mean Income (~80k).")

## 4. Visualizing the Unscaled Space

Let's look at the data in its raw form. When plotted on a standard grid without adjusting the aspect ratio, we can visually see how Income completely dwarfs Age.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Age', y='Income', hue='Bought_Product', alpha=0.7, palette='coolwarm')
plt.title('Raw Unscaled Features: Age vs Income', fontsize=14)
plt.xlabel('Age (Years)', fontsize=12)
plt.ylabel('Income (Dollars)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Visual Check: Any geometric distance calculated here will be 99.9% driven by the Y-axis (Income).")

## 5. Mathematical Approach 1: Min-Max Normalization

Min-Max Normalization rescales the data to a fixed range, typically [0, 1]. This method relies strictly on the minimum and maximum observed values of the feature.

Formula:
Scaled Value = (x - x_min) / (x_max - x_min)

- x_min: Minimum value of the feature.
- x_max: Maximum value of the feature.

Intuition: This is a linear transformation that preserves the original relative relationships between values but maps them precisely into a unit interval.

In [ ]:
# Implementing Min-Max Scaling
min_max_scaler = MinMaxScaler()

# We pass a 2D array by selecting multiple columns
df_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(df[['Age', 'Income']]),
    columns=['Age_MinMax', 'Income_MinMax']
)

print("--- Min-Max Scaled Data Summary ---")
print(df_minmax.describe().round(4))

print("\nInsight: Notice how the minimums are exactly 0.0 and maximums are exactly 1.0 for both features.")

## 6. Mathematical Approach 2: Z-Score Standardization

Z-Score Standardization (also called Standard Scaling) rescales the data to have a mean of 0 and a standard deviation of 1. It relies on the mean (mu) and standard deviation (sigma) of the feature distribution.

Formula:
Scaled Value = (x - mu) / sigma

- mu: Mean of the feature.
- sigma: Standard deviation of the feature.

Intuition: This transforms the data into a standard normal format. It centers the data at zero and scales the dispersion based on the variance.

In [ ]:
# Implementing Z-Score Standardization
standard_scaler = StandardScaler()

df_standard = pd.DataFrame(
    standard_scaler.fit_transform(df[['Age', 'Income']]),
    columns=['Age_Standard', 'Income_Standard']
)

print("--- Standard Scaled Data Summary ---")
print(df_standard.describe().round(4))

print("\nInsight: Notice the means are very close to 0.0, and standard deviations are exactly 1.0.")

## 7. Comparing the Transformations

Now that we have applied both techniques, let's visualize how the distribution shapes remain the same, but the axes have shifted completely.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Raw Data
sns.histplot(df['Income'], ax=axes[0], color='gray', kde=True)
axes[0].set_title('Raw Income')
axes[0].set_xlabel('Income ($)')

# Plot 2: Min-Max Normalization
sns.histplot(df_minmax['Income_MinMax'], ax=axes[1], color='blue', kde=True)
axes[1].set_title('Min-Max Normalized Income')
axes[1].set_xlabel('Scaled Value [0, 1]')

# Plot 3: Z-Score Standardization
sns.histplot(df_standard['Income_Standard'], ax=axes[2], color='green', kde=True)
axes[2].set_title('Z-Score Standardized Income')
axes[2].set_xlabel('Scaled Value (Standard Deviations)')

plt.tight_layout()
plt.show()

print("Observation: The shape of the histogram is perfectly identical across all three plots. Only the x-axis scale changes.")

## 8. The Outlier Trap

What happens if there is an extreme outlier in the dataset? 
Because Min-Max Normalization depends strictly on the absolute min and max, a single extreme outlier will squash all legitimate data into a tiny range, destroying the feature's variance. Standard Scaling handles this much better.

In [ ]:
# Introduce a deliberate massive outlier (e.g., a billionaire)
df_outlier = df.copy()
df_outlier.loc[0, 'Income'] = 5000000  # $5 Million

# Apply both scalers to the outlier dataset
mm_outlier = MinMaxScaler().fit_transform(df_outlier[['Income']])
ss_outlier = StandardScaler().fit_transform(df_outlier[['Income']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show the crushing effect of Min-Max
sns.histplot(mm_outlier, ax=axes[0], bins=30, color='red')
axes[0].set_title('Min-Max with Outlier (Data Squashed)')
axes[0].set_xlabel('Scaled Value')

# Show the resilience of Standard Scaling
sns.histplot(ss_outlier, ax=axes[1], bins=30, color='orange')
axes[1].set_title('Z-Score with Outlier (Better Spread)')
axes[1].set_xlabel('Scaled Value')

plt.tight_layout()
plt.show()

print("Warning: In the left plot, 99.9% of data is crushed into the 0.00 to 0.05 range because the outlier defines the max limit of 1.0.")

## 9. Machine Learning Simulation: KNN Performance

Now let's prove that scaling actually improves model accuracy. We will train a K-Nearest Neighbors (KNN) classifier on three versions of our data:
1. Raw Data
2. Min-Max Scaled Data
3. Standard Scaled Data

In [ ]:
# Setup the Training and Testing Split
X = df[['Age', 'Income']]
y = df['Bought_Product']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

In [ ]:
# Initialize the classifier
knn = KNeighborsClassifier(n_neighbors=5)

# --- Test 1: Raw Data ---
knn.fit(X_train, y_train)
preds_raw = knn.predict(X_test)
acc_raw = accuracy_score(y_test, preds_raw)

# --- Test 2: Min-Max Scaled Data ---
mm_scaler_ml = MinMaxScaler()
X_train_mm = mm_scaler_ml.fit_transform(X_train)
X_test_mm = mm_scaler_ml.transform(X_test)  # Transform test data using train parameters

knn.fit(X_train_mm, y_train)
preds_mm = knn.predict(X_test_mm)
acc_mm = accuracy_score(y_test, preds_mm)

# --- Test 3: Standard Scaled Data ---
std_scaler_ml = StandardScaler()
X_train_std = std_scaler_ml.fit_transform(X_train)
X_test_std = std_scaler_ml.transform(X_test)

knn.fit(X_train_std, y_train)
preds_std = knn.predict(X_test_std)
acc_std = accuracy_score(y_test, preds_std)

# Print Results
print("--- KNN Classifier Accuracy Results ---")
print(f"Raw Data Accuracy:        {acc_raw * 100:.2f}%")
print(f"Min-Max Scaled Accuracy:  {acc_mm * 100:.2f}%")
print(f"Z-Score Scaled Accuracy:  {acc_std * 100:.2f}%")

print("\nProof: Scaling directly enables distance-based algorithms to learn from all available features effectively!")

## 10. Engineering Standard: Avoiding Data Leakage

> **Important rule:** In an ML pipeline, scaling parameters (the mean, standard dev, min, and max) must be treated as part of the model artifact.

**The Trap:** If you calculate the mean or max on the *entire* dataset before the train/test split, you are secretly feeding information about the unseen test data into your training pipeline. This is called Data Leakage.

**The Engineering Standard:** You must `fit` your scalers *only* on the training data, and then `transform` both the training and test datasets using those stored training parameters.

In [ ]:
# --- WRONG APPROACH (Data Leakage) ---
bad_scaler = StandardScaler()
X_all_scaled = bad_scaler.fit_transform(X) # Calculating mu and sigma on EVERYTHING
X_tr_bad, X_te_bad, _, _ = train_test_split(X_all_scaled, y, test_size=0.2, random_state=42)

# --- RIGHT APPROACH (Strict separation) ---
good_scaler = StandardScaler()
X_tr_good, X_te_good, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 1: Fit ONLY on training data (learns the mu and sigma of the train set)
good_scaler.fit(X_tr_good)

# Step 2: Transform both sets using the training parameters
X_tr_final = good_scaler.transform(X_tr_good)
X_te_final = good_scaler.transform(X_te_good)

print("Best Practice Demonstrated: Always separate `fit` from `transform` to protect your test set integrity.")

## 11. Practice Exercises

Let's test our understanding of how feature scaling behaves under different circumstances.

### Exercise 1: Manual Calculation Check

**Task:** Given an array of values `[10, 20, 30, 40, 50]`, write a pure python function that calculates the Min-Max normalized values without using `scikit-learn`.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
values = np.array([10, 20, 30, 40, 50])

def manual_min_max(arr):
    arr_min = np.min(arr)
    arr_max = np.max(arr)
    
    # Apply formula: (x - x_min) / (x_max - x_min)
    scaled = (arr - arr_min) / (arr_max - arr_min)
    return scaled

scaled_values = manual_min_max(values)
print(f"Original Array: {values}")
print(f"Manual Min-Max Scaled: {scaled_values}")

### Exercise 2: When NOT to scale

**Task:** We discussed that Tree-based models (like Decision Trees or Random Forests) do not require feature scaling. Let's run a quick Random Forest test to prove this. Compare the accuracy of a Random Forest on `X_train` (raw) vs `X_train_std` (scaled).

In [ ]:
# --- EXERCISE 2 SOLUTION ---
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42, n_estimators=50)

# Train on raw data
rf.fit(X_train, y_train)
rf_acc_raw = accuracy_score(y_test, rf.predict(X_test))

# Train on scaled data
rf.fit(X_train_std, y_train)
rf_acc_scaled = accuracy_score(y_test, rf.predict(X_test_std))

print(f"Random Forest on Raw Data:    {rf_acc_raw * 100:.2f}%")
print(f"Random Forest on Scaled Data: {rf_acc_scaled * 100:.2f}%")

print("\nProof: Tree models slice the data horizontally/vertically. The absolute scale of the feature axis does not change the splits!")

## 12. Visualization Gallery

To summarize, let's look at how the 2-Dimensional feature space morphs when we apply these different scaling methods. This visualizes the 'spherical' adjustment that makes optimization algorithms (like gradient descent) run faster.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Unscaled Feature Space
sns.scatterplot(x=X_train['Age'], y=X_train['Income'], hue=y_train, alpha=0.6, ax=axes[0])
axes[0].set_title('Raw Features (Elliptical)')

# 2. Min-Max Scaled Space
sns.scatterplot(x=X_train_mm[:, 0], y=X_train_mm[:, 1], hue=y_train, alpha=0.6, ax=axes[1])
axes[1].set_title('Min-Max Space (Bounded Square)')
axes[1].set_xlabel('Age Scaled')
axes[1].set_ylabel('Income Scaled')

# 3. Standard Scaled Space
sns.scatterplot(x=X_train_std[:, 0], y=X_train_std[:, 1], hue=y_train, alpha=0.6, ax=axes[2])
axes[2].set_title('Standardized Space (Spherical)')
axes[2].set_xlabel('Age (Z-Score)')
axes[2].set_ylabel('Income (Z-Score)')

plt.tight_layout()
plt.show()

## 13. Summary and Interview Insights

- **Mandatory for Certain Models:** Scaling is a hard requirement for distance-based models (KNN, SVM, K-Means) and gradient-based models (Logistic Regression, Neural Networks).
- **Min-Max Tradeoffs:** Ideal for bounded data (like images with pixel ranges 0-255) but heavily damaged by extreme outliers.
- **Z-Score Robustness:** Standard Scaling is the preferred default for most ML pipelines because it handles outliers far more gracefully.
- **Leakage Prevention:** Always apply `fit_transform` to the training data alone, and only apply `transform` to validation/test sets using the learned parameters.
- **The Tree Exception:** Decision Trees and Random Forests are invariant to monotonic transformations like scaling, meaning applying these transformations will not affect their predictions.

In [ ]:
print("-----------------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully mastered Feature Scaling and Normalization!")
print("-----------------------------------------------------------")